In [1]:
import json
from pathlib import Path

data_path = Path("../data/products.json")

with data_path.open("r", encoding="utf-8") as file:
    products = json.load(file)

print(f"Loaded {len(products)} products")
print(products[0])


Loaded 1142 products
{'id': 'cl-001', 'name': 'Classic White Cotton T-Shirt', 'category': 't-shirt', 'gender': 'unisex', 'colors': ['white'], 'style': ['casual', 'basic'], 'material': ['cotton'], 'season': ['spring', 'summer'], 'price': 799, 'rating': 4.6, 'popularity': 98, 'sizes': ['S', 'M', 'L', 'XL'], 'stock': 42, 'imageUrl': 'https://images.unsplash.com/photo-1521572163474-6864f9cf17ab?auto=format&fit=crop&w=600&q=80'}


In [2]:
import pandas as pd

products_df = pd.DataFrame(products)

print("Rows and columns:", products_df.shape)
print("Available fields:")
print(products_df.columns.tolist())

products_df.head(3)


Rows and columns: (1142, 14)
Available fields:
['id', 'name', 'category', 'gender', 'colors', 'style', 'material', 'season', 'price', 'rating', 'popularity', 'sizes', 'stock', 'imageUrl']


,id,name,category,gender,colors,style,material,season,price,rating,popularity,sizes,stock,imageUrl
0,cl-001,Classic White Cotton T-Shirt,t-shirt,unisex,[white],"[casual, basic]",[cotton],"[spring, summer]",799,4.6,98,"[S, M, L, XL]",42,https://images.unsplash.com/photo-152157216347...
1,cl-002,Oversized Black Graphic T-Shirt,t-shirt,unisex,[black],"[casual, streetwear, oversized]",[cotton],"[spring, summer, autumn]",999,4.5,94,"[S, M, L, XL]",31,https://images.unsplash.com/photo-150334150425...
2,cl-003,Sky Blue Linen Shirt,shirt,men,[blue],"[casual, smart-casual]",[linen],"[spring, summer]",1599,4.7,91,"[S, M, L, XL]",19,https://images.unsplash.com/photo-160281031838...


In [3]:
products_df.isnull().sum()



id            0
name          0
category      0
gender        0
colors        0
style         0
material      0
season        0
price         0
rating        0
popularity    0
sizes         0
stock         0
imageUrl      0
dtype: int64

In [4]:
feature_columns = [
    "category",
    "gender",
    "colors",
    "style",
    "material",
    "season"
]

def build_feature_text(row):
    parts = []

    for column in feature_columns:
        value = row[column]

        if isinstance(value, list):
            parts.extend(str(item).lower() for item in value)
        else:
            parts.append(str(value).lower())

    return " ".join(parts)

products_df["feature_text"] = products_df.apply(build_feature_text, axis=1)

products_df[["name", "feature_text"]].head()

,name,feature_text
0,Classic White Cotton T-Shirt,t-shirt unisex white casual basic cotton sprin...
1,Oversized Black Graphic T-Shirt,t-shirt unisex black casual streetwear oversiz...
2,Sky Blue Linen Shirt,shirt men blue casual smart-casual linen sprin...
3,Slim Fit Dark Blue Jeans,jeans unisex blue casual everyday denim cotton...
4,High-Waist Straight Blue Jeans,jeans women blue casual everyday high-waist de...


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

feature_matrix = vectorizer.fit_transform(
    products_df["feature_text"]
)

print("Matrix shape:", feature_matrix.shape)

Matrix shape: (1142, 146)


In [6]:
from sklearn.metrics.pairwise import cosine_similarity
similarity_matrix = cosine_similarity(feature_matrix)
print("Similarity matrix shape:", similarity_matrix.shape)

Similarity matrix shape: (1142, 1142)


In [7]:
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=products_df["name"],
    columns=products_df["name"]
)

similarity_df.round(2)

name,Classic White Cotton T-Shirt,Oversized Black Graphic T-Shirt,Sky Blue Linen Shirt,Slim Fit Dark Blue Jeans,High-Waist Straight Blue Jeans,Floral Summer Midi Dress,Elegant Black Evening Dress,Warm Beige Knit Sweater,Olive Lightweight Bomber Jacket,Navy Blue Zip Hoodie,...,Women's Graphic Headband (Headband),Women's Lightweight Headband (Headband),Women's Utility Headband (Headband),Women's Heritage Headband (Headband),Women's Cropped Headband (Headband),Women's Washed Headband (Headband),Women's Signature Headband (Headband),Women's Sport Headband (Headband),Women's Urban Headband (Headband),Women's Weekend Headband (Headband)
name,,,,,,,,,,,,,,,,,,,,,
Classic White Cotton T-Shirt,1.00,0.80,0.52,0.49,0.10,0.11,0.00,0.30,0.06,0.42,...,0.03,0.02,0.08,0.07,0.05,0.05,0.24,0.12,0.13,0.08
Oversized Black Graphic T-Shirt,0.80,1.00,0.49,0.48,0.10,0.07,0.05,0.30,0.16,0.34,...,0.24,0.03,0.09,0.03,0.11,0.05,0.19,0.07,0.38,0.09
Sky Blue Linen Shirt,0.52,0.49,1.00,0.24,0.18,0.11,0.00,0.09,0.16,0.09,...,0.13,0.02,0.05,0.10,0.05,0.05,0.16,0.23,0.02,0.05
Slim Fit Dark Blue Jeans,0.49,0.48,0.24,1.00,0.48,0.07,0.04,0.33,0.11,0.38,...,0.08,0.03,0.21,0.14,0.10,0.15,0.14,0.11,0.08,0.09
High-Waist Straight Blue Jeans,0.10,0.10,0.18,0.48,1.00,0.07,0.05,0.06,0.08,0.08,...,0.08,0.05,0.18,0.13,0.09,0.14,0.13,0.10,0.08,0.09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Women's Washed Headband (Headband),0.05,0.05,0.05,0.15,0.14,0.05,0.07,0.00,0.00,0.00,...,0.30,0.38,0.35,0.30,0.34,1.00,0.35,0.36,0.39,0.45
Women's Signature Headband (Headband),0.24,0.19,0.16,0.14,0.13,0.09,0.09,0.06,0.08,0.17,...,0.37,0.34,0.40,0.32,0.42,0.35,1.00,0.40,0.52,0.41
Women's Sport Headband (Headband),0.12,0.07,0.23,0.11,0.10,0.19,0.11,0.15,0.11,0.09,...,0.35,0.37,0.31,0.39,0.34,0.36,0.40,1.00,0.44,0.32


In [8]:
def recommend_products(product_id, n=5):
    try:
        product_position = products_df["id"].tolist().index(product_id)
    except ValueError:
        raise ValueError(f"Product ID not found: {product_id}")

    scores = similarity_matrix[product_position]

    ranked_positions = scores.argsort()[::-1]

    ranked_positions = [
        position
        for position in ranked_positions
        if position != product_position
    ][:n]

    recommendations = products_df.iloc[ranked_positions][
        ["id", "name", "category", "colors", "style", "price"]
    ].copy()

    recommendations["similarity_score"] = (
        scores[ranked_positions].round(3)
    )

    return recommendations.reset_index(drop=True)

In [9]:
recommend_products("cl-002", n=5)

,id,name,category,colors,style,price,similarity_score
0,cl-001,Classic White Cotton T-Shirt,t-shirt,[white],"[casual, basic]",799,0.797
1,cl-013,Oversized Graphic T-Shirt,t-shirt,"[black, white]","[streetwear, oversized]",999,0.739
2,cl-011,Relaxed Cotton Cargo Pants,trousers,[khaki],"[casual, utility, streetwear]",1999,0.490
3,cl-003,Sky Blue Linen Shirt,shirt,[blue],"[casual, smart-casual]",1599,0.489
4,cl-004,Slim Fit Dark Blue Jeans,jeans,[blue],"[casual, everyday]",2199,0.482


In [10]:
products_df["search_text"] = (
    products_df["name"].fillna("")
    + " "
    + products_df["feature_text"]
)

search_vectorizer = TfidfVectorizer()

search_matrix = search_vectorizer.fit_transform(
    products_df["search_text"]
)

In [11]:
import re


def normalize_search_query(query):
    query = str(query).casefold()
    query = re.sub(r"\s+", " ", query).strip()

    synonyms = {
        "t shirt": "t-shirt",
        "tshirt": "t-shirt",
        "tee": "t-shirt",
        "tees": "t-shirt",
        "pant": "pants",
        "trouser": "trousers",
        "jean": "jeans",
        "hoodies": "hoodie",
        "shirts": "shirt",
        "jackets": "jacket",
        "sweaters": "sweater",
        "dresses": "dress",
        "dark blue": "dark-blue",
        "light blue": "light-blue",
        "navy blue": "navy"
    }

    for original, standard in sorted(
        synonyms.items(),
        key=lambda item: len(item[0]),
        reverse=True
    ):
        query = re.sub(
            rf"(?<![a-z0-9]){re.escape(original)}(?![a-z0-9])",
            standard,
            query
        )

    return re.sub(r"\s+", " ", query).strip()

In [12]:
import numpy as np


def _column_values(column_name):
    values = set()

    for value in products_df[column_name].dropna():
        if isinstance(value, list):
            values.update(
                str(item).casefold().strip()
                for item in value
            )
        else:
            values.add(str(value).casefold().strip())

    return {value for value in values if value}


def _query_contains(query, phrase):
    return re.search(
        rf"(?<![a-z0-9]){re.escape(phrase)}(?![a-z0-9])",
        query
    ) is not None


def _row_contains_any(values, requested_values):
    if not isinstance(values, list):
        values = [values]

    actual_values = {
        str(value).casefold().strip()
        for value in values
    }

    return bool(actual_values.intersection(requested_values))


def extract_query_features(query):
    normalized_query = normalize_search_query(query)

    category_aliases = {
        "pants": {"pants", "jeans", "trousers"},
        "jeans": {"jeans"},
        "trousers": {"trousers"},
        "t-shirt": {"t-shirt", "t-shirts"},
        "shirt": {"shirt", "shirts"},
        "hoodie": {"hoodie", "hoodies"},
        "sweater": {"sweater", "sweaters"},
        "jacket": {"jacket", "jackets"},
        "dress": {"dress", "dresses"},
        "lounge-set": {"lounge-set", "lounge set"}
    }

    category = None
    for alias in sorted(category_aliases, key=len, reverse=True):
        if _query_contains(normalized_query, alias):
            category = alias
            break

    known_colors = _column_values("colors")
    known_styles = _column_values("style")
    known_materials = _column_values("material")
    known_seasons = _column_values("season")
    known_genders = _column_values("gender")

    colors = {
        value for value in known_colors
        if _query_contains(normalized_query, value)
    }
    styles = {
        value for value in known_styles
        if _query_contains(normalized_query, value)
    }
    materials = {
        value for value in known_materials
        if _query_contains(normalized_query, value)
    }
    seasons = {
        value for value in known_seasons
        if _query_contains(normalized_query, value)
    }
    genders = {
        value for value in known_genders
        if _query_contains(normalized_query, value)
    }

    price_match = re.search(
        r"(?:under|below|less than|up to|upto)\s*"
        r"(?:₹|rs\.?|inr|\$)?\s*([\d,]+)",
        normalized_query
    )
    max_price = (
        int(price_match.group(1).replace(",", ""))
        if price_match
        else None
    )

    return {
        "normalized_query": normalized_query,
        "category": category,
        "colors": colors,
        "styles": styles,
        "materials": materials,
        "seasons": seasons,
        "genders": genders,
        "max_price": max_price
    }


def search_products(query, n=50):
    if search_matrix.shape[0] != len(products_df):
        raise RuntimeError(
            "Catalog and search matrix are out of sync. "
            "Restart the kernel and run all cells."
        )

    features = extract_query_features(query)
    normalized_query = features["normalized_query"]

    query_vector = search_vectorizer.transform(
        [normalized_query]
    )

    if query_vector.nnz > 0:
        scores = cosine_similarity(
            query_vector,
            search_matrix
        ).ravel()
    else:
        scores = np.zeros(len(products_df))

    in_stock = products_df["stock"].fillna(0).gt(0)
    exact_mask = in_stock.copy()
    has_structured_filter = False

    category_groups = {
        "pants": {"jeans", "trousers", "pants"},
        "jeans": {"jeans"},
        "trousers": {"trousers"},
        "t-shirt": {"t-shirt", "t-shirts"},
        "shirt": {"shirt", "shirts"},
        "hoodie": {"hoodie", "hoodies"},
        "sweater": {"sweater", "sweaters"},
        "jacket": {"jacket", "jackets"},
        "dress": {"dress", "dresses"},
        "lounge-set": {"lounge-set", "lounge set"}
    }

    category_values = (
        products_df["category"]
        .astype(str)
        .str.casefold()
        .str.strip()
    )

    if features["category"]:
        exact_mask &= category_values.isin(
            category_groups[features["category"]]
        )
        has_structured_filter = True

    for column_name, requested_values in [
        ("colors", features["colors"]),
        ("style", features["styles"]),
        ("material", features["materials"]),
        ("season", features["seasons"])
    ]:
        if requested_values:
            exact_mask &= products_df[column_name].apply(
                lambda values: _row_contains_any(
                    values,
                    requested_values
                )
            )
            has_structured_filter = True

    if features["genders"]:
        gender_values = (
            products_df["gender"]
            .astype(str)
            .str.casefold()
            .str.strip()
        )
        exact_mask &= gender_values.isin(features["genders"])
        has_structured_filter = True

    if features["max_price"] is not None:
        exact_mask &= products_df["price"].fillna(np.inf).le(
            features["max_price"]
        )
        has_structured_filter = True

    if has_structured_filter and exact_mask.any():
        candidate_mask = exact_mask
        match_type = "exact"
    else:
        candidate_mask = in_stock & (scores > 0)
        match_type = "related"

        if features["max_price"] is not None:
            candidate_mask &= products_df["price"].fillna(np.inf).le(
                features["max_price"]
            )

    if not candidate_mask.any():
        return pd.DataFrame()

    results = products_df.loc[candidate_mask].copy()
    results["search_score"] = scores[candidate_mask.to_numpy()]
    results["match_type"] = match_type

    results = results.sort_values(
        ["search_score", "popularity"],
        ascending=[False, False]
    )

    return results[
        [
            "id",
            "name",
            "category",
            "colors",
            "style",
            "price",
            "stock",
            "search_score",
            "match_type"
        ]
    ].head(n).reset_index(drop=True)

In [13]:
def _canonical_category(value):
    value = str(value).casefold().strip()

    aliases = {
        "tshirt": "t-shirt",
        "t-shirts": "t-shirt",
        "t shirt": "t-shirt",
        "shirts": "shirt",
        "hoodies": "hoodie",
        "sweaters": "sweater",
        "jackets": "jacket",
        "dresses": "dress",
        "trouser": "trousers",
        "pants": "pants",
        "jean": "jeans"
    }

    return aliases.get(value, value)


def recommend_products(
    product_id,
    n=5,
    gender=None,
    category=None,
    include_unisex=True
):
    product_ids = products_df["id"].tolist()

    if product_id not in product_ids:
        raise ValueError(f"Product ID not found: {product_id}")

    product_position = product_ids.index(product_id)
    selected_product = products_df.iloc[product_position]

    category_groups = {
        "pants": {"pants", "trousers", "jeans"},
        "jeans": {"jeans"},
        "trousers": {"trousers"},
        "t-shirt": {"t-shirt", "t-shirts"},
        "shirt": {"shirt", "shirts"},
        "hoodie": {"hoodie", "hoodies"},
        "sweater": {"sweater", "sweaters"},
        "jacket": {"jacket", "jackets"},
        "dress": {"dress", "dresses"},
        "lounge-set": {"lounge-set", "lounge set"}
    }

    # If no category is provided, use the selected product's category
    requested_category = _canonical_category(
        category if category is not None
        else selected_product["category"]
    )

    allowed_categories = category_groups.get(
        requested_category,
        {requested_category}
    )

    category_values = (
        products_df["category"]
        .astype(str)
        .str.casefold()
        .str.strip()
    )

    candidate_mask = (
        category_values.isin(allowed_categories)
        & products_df["stock"].fillna(0).gt(0)
    )

    # Apply the selected gender page
    if gender is not None:
        requested_gender = str(gender).casefold().strip()

        allowed_genders = {requested_gender}

        if include_unisex and requested_gender in {"men", "women"}:
            allowed_genders.add("unisex")

        gender_values = (
            products_df["gender"]
            .astype(str)
            .str.casefold()
            .str.strip()
        )

        candidate_mask &= gender_values.isin(allowed_genders)

    # Do not recommend the product currently being viewed
    candidate_mask.iloc[product_position] = False

    if not candidate_mask.any():
        return pd.DataFrame()

    scores = similarity_matrix[product_position]

    ranked_positions = [
        int(position)
        for position in scores.argsort()[::-1]
        if candidate_mask.iloc[int(position)]
    ][:n]

    recommendations = products_df.iloc[ranked_positions][
        [
            "id",
            "name",
            "category",
            "gender",
            "colors",
            "style",
            "price",
            "stock"
        ]
    ].copy()

    recommendations["similarity_score"] = (
        scores[ranked_positions].round(3)
    )

    return recommendations.reset_index(drop=True)

In [19]:
selected_id = "cl-599"

recommendations = recommend_products(
    selected_id,
    n=5
)

recommendations


,id,name,category,gender,colors,style,price,stock,similarity_score
0,cl-105,Men's Cropped T-Shirt (T-Shirts),T-Shirts,men,"[red, cream]","[sporty, athleisure]",899,82,0.560
1,cl-591,Women's Essential T-Shirt (T-Shirts),T-Shirts,women,"[black, white]","[utility, workwear]",1199,55,0.555
2,cl-607,Women's Utility T-Shirt (T-Shirts),T-Shirts,women,"[maroon, charcoal]","[utility, workwear]",799,7,0.548
3,cl-610,Women's Washed T-Shirt (T-Shirts),T-Shirts,women,"[red, cream]","[smart-casual, minimal]",1099,34,0.484
4,cl-609,Women's Cropped T-Shirt (T-Shirts),T-Shirts,women,"[green, black]","[casual, basic]",1199,25,0.461


In [16]:
search_products("red tshirt")

,id,name,category,colors,style,price,stock,search_score,match_type
0,cl-599,Women's Everyday T-Shirt (T-Shirts),T-Shirts,"[red, cream]","[utility, workwear]",999,31,0.421270,exact
1,cl-094,Men's Oversized T-Shirt (T-Shirts),T-Shirts,"[red, cream]","[casual, basic]",799,79,0.417883,exact
2,cl-105,Men's Cropped T-Shirt (T-Shirts),T-Shirts,"[red, cream]","[sporty, athleisure]",899,82,0.412815,exact
3,cl-610,Women's Washed T-Shirt (T-Shirts),T-Shirts,"[red, cream]","[smart-casual, minimal]",1099,34,0.386512,exact
